In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q03/q03_rewrite/checkpoints/pre_cell_3.pickle

In [ ]:
%%cudf.pandas.profile
### cell 3 ###

# Define the cutoff date as a pandas Timestamp (will be handled on GPU)
date = pd.Timestamp("1995-03-04")

# 1) Filter rows and select only the needed columns, all on GPU
fcustomer = customer.loc[
    customer.C_MKTSEGMENT == "HOUSEHOLD",
    ["C_CUSTKEY"]
]

forders = orders.loc[
    orders.O_ORDERDATE < date,
    ["O_ORDERKEY", "O_CUSTKEY", "O_ORDERDATE", "O_SHIPPRIORITY"]
]

flineitem = lineitem.loc[
    lineitem.L_SHIPDATE > date,
    ["L_ORDERKEY", "L_EXTENDEDPRICE", "L_DISCOUNT"]
]

# 2) Perform the two joins entirely on GPU
jn2 = (
    fcustomer
      .merge(forders,    left_on="C_CUSTKEY", right_on="O_CUSTKEY")
      .merge(flineitem,  left_on="O_ORDERKEY", right_on="L_ORDERKEY")
)

# 3) Compute the TMP column, group, sum and sort—all GPU operations
jn2["TMP"] = jn2.L_EXTENDEDPRICE * (1 - jn2.L_DISCOUNT)

res = (
    jn2
      .groupby(
          ["L_ORDERKEY", "O_ORDERDATE", "O_SHIPPRIORITY"],
          as_index=False,
          sort=False
      )["TMP"]
      .sum()
      .sort_values("TMP", ascending=False)
)

# res now has the same schema [L_ORDERKEY, TMP, O_ORDERDATE, O_SHIPPRIORITY]
